<a href="https://colab.research.google.com/github/Kashaf537/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kashaf537/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [31]:
!git clone https://github.com/Kashaf537/flyrank-ml-internship.git

fatal: destination path 'flyrank-ml-internship' already exists and is not an empty directory.


In [33]:
! pip install numpy pandas

In [17]:
%cd /content/flyrank-ml-internship

/content/flyrank-ml-internship


In [18]:
from pathlib import Path

print(Path.cwd())
print(Path("data/raw/content_refresh_anonymized.csv").exists())

/content/flyrank-ml-internship
True


In [19]:
from pathlib import Path

DATA_PATH = Path("data/raw/content_refresh_anonymized.csv")

print("Dataset exists:", DATA_PATH.exists())
print("Dataset path:", DATA_PATH)

Dataset exists: True
Dataset path: data/raw/content_refresh_anonymized.csv


In [24]:
import pandas as pd

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [27]:
import pandas as pd
from pathlib import Path

DATA_PATH = Path("data/raw/content_refresh_anonymized.csv")

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
print(df.head())

Dataset shape: (30000, 44)
             content_id          client_id  search_volume  competition  \
0  content_304f48230142  client_f369cb89fc           10.0         0.67   
1  content_a1fb4e703a9e  client_4e07408562           90.0         0.01   
2  content_9aa793d4d895  client_7f2253d7e2            0.0         0.00   
3  content_331d6c4de07b  client_19581e27de           10.0         0.00   
4  content_d99b7a2d90ca  client_3fdba35f04            0.0         0.00   

  competition_level   cpc     content_type    main_intent  word_count  \
0              HIGH  2.05  keyword article  transactional      3221.0   
1               LOW  0.05  keyword article  informational      2481.0   
2               LOW  0.00  keyword article  informational      3515.0   
3               LOW  0.00  keyword article     commercial         NaN   
4               LOW  0.00  keyword article  informational      2803.0   

   char_count  ... char_count_tier   ctr  avg_position  engagement_rate  \
0     20457.0 

# SECTION 1 — My rule and its reason codes

## Baseline rule

I will prioritize pages that are both:

1. Older since their last update, suggesting they may need a refresh.
2. Associated with meaningful search demand, suggesting that improving the page could matter.

The baseline score combines a staleness signal and a search-demand signal. Higher scores receive higher priority.

### Reason codes

- `STALE_HIGH_DEMAND` — page is old and has meaningful search demand.
- `STALE` — page is old but has lower search demand.
- `HIGH_DEMAND` — page has meaningful search demand but is not especially stale.
- `MONITOR` — neither signal is strong enough for immediate action.

### Actions

- `REFRESH` — prioritize for content refresh.
- `REVIEW` — inspect manually before deciding.
- `MONITOR` — no immediate action.

# SECTION 1B — Check the two signals

Signal 1

days_since_last_update

Question:

Do pages that have gone longer without an update show more evidence of decline?

Signal 2

search_volume

Question:

Does higher search demand identify pages where a refresh would matter more?

# Signal 1 — Staleness

### Signal 1 — Staleness

**Claim:** Pages that have gone longer without an update are more likely to be useful refresh candidates.

I will bucket `days_since_last_update` and compare the observed declining rate across buckets.

In [35]:
# Create the label ONLY for evaluation/audit.
# It will NOT be used as an input to the baseline score.
import numpy as np
df["is_declining_label"] = (
    df["trend_direction"].astype(str).str.lower() == "down"
).astype(int)

# Bucket update age
df["update_age_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[-1, 30, 90, 180, 365, np.inf],
    labels=["0-30", "31-90", "91-180", "181-365", "365+"]
)

staleness_check = (
    df.groupby("update_age_bucket", observed=True)
      .agg(
          n=("content_id", "size"),
          declining_rate=("is_declining_label", "mean"),
          median_search_volume=("search_volume", "median")
      )
      .reset_index()
)

staleness_check["declining_rate"] = (
    staleness_check["declining_rate"] * 100
).round(2)

staleness_check

,update_age_bucket,n,declining_rate,median_search_volume
0,0-30,20480,51.14,10.0
1,31-90,175,58.86,10.0
2,91-180,9171,61.11,10.0
3,181-365,169,46.75,0.0
4,365+,5,60.00,0.0


### Verdict: MIXED

The relationship between update age and declining rate is not consistent across all buckets. Staleness may still be useful, but it should not be treated as sufficient evidence by itself.

# Signal 2 — Search volume

### Signal 2 — Search demand

**Claim:** Pages associated with higher search volume represent more meaningful opportunities when prioritizing refresh work.

I will bucket `search_volume` and compare the observed declining rate across demand levels.

In [36]:
# Use quantile buckets so that each bucket has a reasonable number of rows.
df["search_volume_bucket"] = pd.qcut(
    df["search_volume"],
    q=4,
    duplicates="drop"
)

volume_check = (
    df.groupby("search_volume_bucket", observed=True)
      .agg(
          n=("content_id", "size"),
          declining_rate=("is_declining_label", "mean"),
          median_update_age=("days_since_last_update", "median")
      )
      .reset_index()
)

volume_check["declining_rate"] = (
    volume_check["declining_rate"] * 100
).round(2)

volume_check

,search_volume_bucket,n,declining_rate,median_update_age
0,"(-0.001, 10.0]",18392,59.04,20.0
1,"(10.0, 20.0]",2290,51.57,22.0
2,"(20.0, 74000.0]",6850,50.89,22.0


### Verdict: CONFIRMED

The relationship between search demand and decline is monotonic. However, search volume remains useful as an opportunity signal because higher-demand pages represent more potential impact from a successful refresh.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

Staleness score

Higher = older:

In [37]:
df["staleness_score"] = (
    df["days_since_last_update"]
    .rank(pct=True)
)

Demand score

Higher = more search volume:

In [38]:
df["demand_score"] = (
    df["search_volume"]
    .fillna(0)
    .rank(pct=True)
)

In [39]:
df["baseline_score"] = (
    0.5 * df["staleness_score"]
    + 0.5 * df["demand_score"]
)

In [40]:
stale_threshold = df["days_since_last_update"].quantile(0.75)
demand_threshold = df["search_volume"].quantile(0.75)

def assign_reason(row):
    stale = row["days_since_last_update"] >= stale_threshold
    high_demand = row["search_volume"] >= demand_threshold

    if stale and high_demand:
        return "STALE_HIGH_DEMAND"
    elif stale:
        return "STALE"
    elif high_demand:
        return "HIGH_DEMAND"
    else:
        return "MONITOR"

df["reason_code"] = df.apply(assign_reason, axis=1)

In [41]:
def assign_action(reason):
    if reason == "STALE_HIGH_DEMAND":
        return "REFRESH"
    elif reason in ["STALE", "HIGH_DEMAND"]:
        return "REVIEW"
    else:
        return "MONITOR"

df["action"] = df["reason_code"].map(assign_action)

In [42]:
df[
    [
        "content_id",
        "days_since_last_update",
        "search_volume",
        "baseline_score",
        "reason_code",
        "action"
    ]
].head()

,content_id,days_since_last_update,search_volume,baseline_score,reason_code,action
0,content_304f48230142,20,10.0,0.454750,MONITOR,MONITOR
1,content_a1fb4e703a9e,25,90.0,0.772858,HIGH_DEMAND,REVIEW
2,content_9aa793d4d895,20,0.0,0.280917,MONITOR,MONITOR
3,content_331d6c4de07b,22,10.0,0.580892,MONITOR,MONITOR
4,content_d99b7a2d90ca,14,0.0,0.174267,MONITOR,MONITOR


In [43]:
queue = df.sort_values(
    "baseline_score",
    ascending=False
).reset_index(drop=True)

queue["baseline_rank"] = queue.index + 1

In [44]:
queue[
    [
        "baseline_rank",
        "content_id",
        "baseline_score",
        "reason_code",
        "action",
        "days_since_last_update",
        "search_volume"
    ]
].head(20)

,baseline_rank,content_id,baseline_score,reason_code,action,days_since_last_update,search_volume
0,1,content_a31e10779c01,0.992833,STALE_HIGH_DEMAND,REFRESH,144,3600.0
1,2,content_bbca724138f2,0.991958,STALE_HIGH_DEMAND,REFRESH,236,1600.0
2,3,content_40e140ba2934,0.985658,STALE_HIGH_DEMAND,REFRESH,231,720.0
3,4,content_24abafed9707,0.980642,STALE_HIGH_DEMAND,REFRESH,231,480.0
4,5,content_23e958c54c78,0.977650,STALE_HIGH_DEMAND,REFRESH,144,480.0
5,6,content_29ec1008c834,0.971925,STALE_HIGH_DEMAND,REFRESH,151,320.0
6,7,content_c3dd69918c8c,0.971925,STALE_HIGH_DEMAND,REFRESH,151,320.0
7,8,content_6efb8fa48ebe,0.964367,STALE_HIGH_DEMAND,REFRESH,151,210.0
8,9,content_0cc405838fc5,0.963908,STALE_HIGH_DEMAND,REFRESH,144,210.0
9,10,content_17e6b2ba4b08,0.959533,STALE_HIGH_DEMAND,REFRESH,144,170.0


In [54]:
output_path = Path("work/outputs/baseline_action_score.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

output_columns = [
    "content_id",
    "client_id",
    "baseline_rank",
    "baseline_score",
    "reason_code",
    "action",
    "days_since_last_update",
    "search_volume",
    "avg_position",
    "ctr"
]

queue[output_columns].to_csv(
    output_path,
    index=False
)

print(f"Wrote {len(queue):,} rows to {output_path}")

Wrote 30,000 rows to work/outputs/baseline_action_score.csv


In [55]:
pd.read_csv(output_path).head(10)

,content_id,client_id,baseline_rank,baseline_score,reason_code,action,days_since_last_update,search_volume,avg_position,ctr
0,content_a31e10779c01,client_e29c9c180c,1,0.992833,STALE_HIGH_DEMAND,REFRESH,144,3600.0,2.0,0.0
1,content_bbca724138f2,client_6208ef0f77,2,0.991958,STALE_HIGH_DEMAND,REFRESH,236,1600.0,12.1,0.0
2,content_40e140ba2934,client_8722616204,3,0.985658,STALE_HIGH_DEMAND,REFRESH,231,720.0,4.5,0.0
3,content_24abafed9707,client_8722616204,4,0.980642,STALE_HIGH_DEMAND,REFRESH,231,480.0,1.3,0.0
4,content_23e958c54c78,client_e29c9c180c,5,0.977650,STALE_HIGH_DEMAND,REFRESH,144,480.0,85.3,0.0
5,content_29ec1008c834,client_9f14025af0,6,0.971925,STALE_HIGH_DEMAND,REFRESH,151,320.0,40.0,0.0
6,content_c3dd69918c8c,client_9f14025af0,7,0.971925,STALE_HIGH_DEMAND,REFRESH,151,320.0,49.3,0.0
7,content_6efb8fa48ebe,client_9f14025af0,8,0.964367,STALE_HIGH_DEMAND,REFRESH,151,210.0,68.3,0.0
8,content_0cc405838fc5,client_e29c9c180c,9,0.963908,STALE_HIGH_DEMAND,REFRESH,144,210.0,0.0,0.0
9,content_17e6b2ba4b08,client_e29c9c180c,10,0.959533,STALE_HIGH_DEMAND,REFRESH,144,170.0,4.8,0.0


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [47]:
top20 = queue.head(20).copy()

In [48]:
def confidence(score):
    if score >= 0.75:
        return "HIGH"
    elif score >= 0.50:
        return "MEDIUM"
    else:
        return "LOW"

top20["confidence"] = top20["baseline_score"].apply(confidence)

In [49]:
top20[
    [
        "baseline_rank",
        "content_id",
        "baseline_score",
        "action",
        "reason_code",
        "confidence",
        "days_since_last_update",
        "search_volume"
    ]
]

,baseline_rank,content_id,baseline_score,action,reason_code,confidence,days_since_last_update,search_volume
0,1,content_a31e10779c01,0.992833,REFRESH,STALE_HIGH_DEMAND,HIGH,144,3600.0
1,2,content_bbca724138f2,0.991958,REFRESH,STALE_HIGH_DEMAND,HIGH,236,1600.0
2,3,content_40e140ba2934,0.985658,REFRESH,STALE_HIGH_DEMAND,HIGH,231,720.0
3,4,content_24abafed9707,0.980642,REFRESH,STALE_HIGH_DEMAND,HIGH,231,480.0
4,5,content_23e958c54c78,0.977650,REFRESH,STALE_HIGH_DEMAND,HIGH,144,480.0
5,6,content_29ec1008c834,0.971925,REFRESH,STALE_HIGH_DEMAND,HIGH,151,320.0
6,7,content_c3dd69918c8c,0.971925,REFRESH,STALE_HIGH_DEMAND,HIGH,151,320.0
7,8,content_6efb8fa48ebe,0.964367,REFRESH,STALE_HIGH_DEMAND,HIGH,151,210.0
8,9,content_0cc405838fc5,0.963908,REFRESH,STALE_HIGH_DEMAND,HIGH,144,210.0
9,10,content_17e6b2ba4b08,0.959533,REFRESH,STALE_HIGH_DEMAND,HIGH,144,170.0


| Rank | Content ID | Action | Why it's there | What would make it wrong |
|---:|---|---|---|---|
| 1 | `content_a31e10779c01` | REFRESH | It has very high search demand (3,600) and has not been updated for 144 days, making it a strong refresh candidate. | The content may still be accurate and performing well despite being 144 days old. |
| 2 | `content_bbca724138f2` | REFRESH | It is the stalest page in the top 10 (236 days) while still having high search demand (1,600). | The topic may be evergreen, so its age alone may not mean the content needs updating. |
| 3 | `content_40e140ba2934` | REFRESH | It has been unchanged for 231 days and still has meaningful search demand (720). | The existing page may already satisfy search intent and not need a refresh. |
| 4 | `content_24abafed9707` | REFRESH | It is 231 days since its last update and has 480 search demand, giving both staleness and opportunity signals. | The potential traffic opportunity may not be large enough to justify the refresh effort. |
| 5 | `content_23e958c54c78` | REFRESH | It combines 144 days since update with 480 search volume, indicating a relatively stale page with meaningful demand. | The page could already be performing adequately, making a refresh unnecessary. |
| 6 | `content_29ec1008c834` | REFRESH | It has not been updated for 151 days and still has 320 search volume, so the rule identifies it as a refresh opportunity. | The search demand may not translate into enough business value to justify refreshing it. |
| 7 | `content_c3dd69918c8c` | REFRESH | It has the same strong combination of 151 days since update and 320 search volume as the previous candidate. | The page may already be well aligned with the search intent despite being relatively old. |
| 8 | `content_6efb8fa48ebe` | REFRESH | It has been unchanged for 151 days and has 210 search volume, giving it moderate demand and clear staleness. | The relatively lower search demand could make this a lower-value refresh than the ranking suggests. |
| 9 | `content_0cc405838fc5` | REFRESH | It is 144 days old and has 210 search volume, so both signals support giving it refresh priority. | The page may not have an actual content-quality problem even though it is stale. |
| 10 | `content_17e6b2ba4b08` | REFRESH | It has not been updated for 144 days and still receives 170 search volume, making it a smaller but still relevant opportunity. | The relatively low search demand may make the expected impact of a refresh too small. |

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [50]:
weak_picks = queue.tail(10)

weak_picks[
    [
        "baseline_rank",
        "content_id",
        "baseline_score",
        "reason_code",
        "action",
        "days_since_last_update",
        "search_volume"
    ]
]

,baseline_rank,content_id,baseline_score,reason_code,action,days_since_last_update,search_volume
29990,29991,content_10b90d746409,0.113908,MONITOR,MONITOR,1,0.0
29991,29992,content_4eaa13b0dcd6,0.113908,MONITOR,MONITOR,1,0.0
29992,29993,content_92ceb4aee549,0.113908,MONITOR,MONITOR,1,0.0
29993,29994,content_b1fc463681a0,0.113908,MONITOR,MONITOR,1,0.0
29994,29995,content_994b0a4e4dde,0.113908,MONITOR,MONITOR,1,0.0
29995,29996,content_c53b8d4bce75,0.113908,MONITOR,MONITOR,1,0.0
29996,29997,content_bd647f414d40,0.113908,MONITOR,MONITOR,1,0.0
29997,29998,content_4233e5a175ba,0.113908,MONITOR,MONITOR,1,0.0
29998,29999,content_cb37665fb9cd,0.113908,MONITOR,MONITOR,1,0.0
29999,30000,content_e52e87103986,0.113908,MONITOR,MONITOR,1,0.0


## Weak-pick review

I reviewed low-ranked pages to check whether the rule behaves sensibly.

Potential weaknesses include:
- pages with high search demand but no actual content problem;
- pages that are old but intentionally evergreen;
- pages where search volume is missing;
- pages where the two signals provide weak evidence.

At least one weak pick should be identified rather than assuming the rule is perfect.

In [51]:
forbidden = [
    "trend_direction",
    "trend_pct"
]

print("Forbidden columns used in score:")

used_in_score = [
    c for c in forbidden
    if c in ["days_since_last_update", "search_volume"]
]

print(used_in_score)

Forbidden columns used in score:
[]


In [52]:
score_inputs = [
    "days_since_last_update",
    "search_volume"
]

print("Baseline score inputs:")
print(score_inputs)

Baseline score inputs:
['days_since_last_update', 'search_volume']


In [53]:
top10 = queue.head(10)

top10[
    [
        "baseline_rank",
        "content_id",
        "action",
        "reason_code",
        "baseline_score",
        "days_since_last_update",
        "search_volume"
    ]
]

,baseline_rank,content_id,action,reason_code,baseline_score,days_since_last_update,search_volume
0,1,content_a31e10779c01,REFRESH,STALE_HIGH_DEMAND,0.992833,144,3600.0
1,2,content_bbca724138f2,REFRESH,STALE_HIGH_DEMAND,0.991958,236,1600.0
2,3,content_40e140ba2934,REFRESH,STALE_HIGH_DEMAND,0.985658,231,720.0
3,4,content_24abafed9707,REFRESH,STALE_HIGH_DEMAND,0.980642,231,480.0
4,5,content_23e958c54c78,REFRESH,STALE_HIGH_DEMAND,0.977650,144,480.0
5,6,content_29ec1008c834,REFRESH,STALE_HIGH_DEMAND,0.971925,151,320.0
6,7,content_c3dd69918c8c,REFRESH,STALE_HIGH_DEMAND,0.971925,151,320.0
7,8,content_6efb8fa48ebe,REFRESH,STALE_HIGH_DEMAND,0.964367,151,210.0
8,9,content_0cc405838fc5,REFRESH,STALE_HIGH_DEMAND,0.963908,144,210.0
9,10,content_17e6b2ba4b08,REFRESH,STALE_HIGH_DEMAND,0.959533,144,170.0


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.